In [0]:
from pyspark.sql import functions as F

login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"
silver = f"{catalog}.{login}_silver"
gold = f"{catalog}.{login}_gold"

def st(name):
    return f"{silver}.olist_{name}_silver"

def gt(name):
    return f"{gold}.{name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

### Star schema

In [0]:
bounds = spark.sql(f"SELECT min(to_date(purchase_ts)) AS lo, max(to_date(purchase_ts)) AS hi FROM {st('orders')}").first()
lo, hi = str(bounds["lo"]), str(bounds["hi"])
print("date range:", lo, hi)

spark.sql(f"""
CREATE OR REPLACE TABLE {gt('dim_date')} COMMENT 'Calendar dimension, one row per day' AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS date,
  year(d) AS year,
  quarter(d) AS quarter,
  month(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day,
  dayofweek(d) AS day_of_week,
  date_format(d, 'EEEE') AS day_name,
  dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(DATE'{lo}', DATE'{hi}', INTERVAL 1 DAY)) AS d)
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gt('dim_customer')} COMMENT 'Customer dimension (one row per customer_id)' AS
SELECT customer_id, customer_unique_id, zip_code_prefix, city, state
FROM {st('customers')}
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gt('dim_product')} COMMENT 'Product dimension with English category name' AS
SELECT
  product_id,
  category_name AS category_pt,
  coalesce(category_name_en, category_name, 'unknown') AS category,
  weight_g, length_cm, height_cm, width_cm, photos_qty
FROM {st('products')}
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gt('dim_seller')} COMMENT 'Seller dimension, current SCD2 version only' AS
SELECT seller_id, zip_code_prefix, city, state
FROM {st('sellers')}
WHERE is_current
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gt('fact_order_items')}
COMMENT 'Fact table, grain = one order item. Measures: price, freight_value, total_value, delivery_days, review_score'
AS
WITH rev AS (
  SELECT order_id, avg(review_score) AS review_score
  FROM {st('reviews')}
  GROUP BY order_id
)
SELECT
  i.order_id,
  i.order_item_id,
  CAST(date_format(o.purchase_ts, 'yyyyMMdd') AS INT) AS date_key,
  to_date(o.purchase_ts) AS order_date,
  o.customer_id,
  c.state AS customer_state,
  i.product_id,
  i.seller_id,
  o.order_status,
  i.price,
  i.freight_value,
  i.price + i.freight_value AS total_value,
  datediff(o.delivered_customer_ts, o.purchase_ts) AS delivery_days,
  CASE WHEN o.delivered_customer_ts IS NULL THEN NULL
       ELSE o.delivered_customer_ts > o.estimated_delivery_ts END AS is_late,
  r.review_score,
  current_timestamp() AS _gold_updated_at
FROM {st('order_items')} i
JOIN {st('orders')} o ON i.order_id = o.order_id
LEFT JOIN {st('customers')} c ON o.customer_id = c.customer_id
LEFT JOIN rev r ON i.order_id = r.order_id
""")

Sanity checks: grain is unique, nothing lost silently:

In [0]:
fact = spark.table(gt("fact_order_items"))
dups = fact.groupBy("order_id", "order_item_id").count().filter("count > 1").count()
silver_items = spark.table(st("order_items")).count()
orphans = (spark.table(st("order_items")).join(spark.table(st("orders")), "order_id", "left_anti").count())
print("duplicate grain keys:", dups)
print("silver items:", silver_items, "| fact rows:", fact.count(), "| items without order:", orphans)
print("fact rows == silver items - orphans:", fact.count() == silver_items - orphans)

rows = [(t, spark.table(gt(t)).count()) for t in ["dim_date", "dim_customer", "dim_product", "dim_seller", "fact_order_items"]]
display(spark.createDataFrame(rows, "table string, rows long"))

### Aggregations:

Canceled and unavailable orders are excluded from revenue.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {gt('agg_sales_daily_state')} COMMENT 'Daily sales by customer state, excl. canceled/unavailable' AS
SELECT
  order_date,
  customer_state,
  count(DISTINCT order_id) AS orders,
  count(*) AS items,
  sum(price) AS revenue,
  sum(freight_value) AS freight
FROM {gt('fact_order_items')}
WHERE order_status NOT IN ('canceled', 'unavailable')
GROUP BY order_date, customer_state
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {gt('agg_sales_monthly_category_state')}
COMMENT 'Monthly sales, delivery and review measures by category and customer state' AS
SELECT
  date_trunc('month', f.order_date) AS month_start,
  f.customer_state,
  p.category,
  count(*) AS items,
  sum(f.price) AS revenue,
  sum(f.freight_value) AS freight,
  count(f.is_late) AS delivered_items,
  sum(CASE WHEN f.is_late THEN 1 ELSE 0 END) AS late_items,
  sum(f.review_score) AS review_sum,
  count(f.review_score) AS review_cnt
FROM {gt('fact_order_items')} f
LEFT JOIN {gt('dim_product')} p ON f.product_id = p.product_id
WHERE f.order_status NOT IN ('canceled', 'unavailable')
GROUP BY date_trunc('month', f.order_date), f.customer_state, p.category
""")

display(spark.table(gt("agg_sales_daily_state")).orderBy(F.desc("revenue")).limit(10))

### Volume monitor and simulated drop:

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gt('ops_volume_log')} (
  snapshot_ts TIMESTAMP, table_name STRING, row_count BIGINT, note STRING
)""")

def snapshot(note):
    n = spark.table(gt("fact_order_items")).count()
    spark.sql(f"INSERT INTO {gt('ops_volume_log')} VALUES (current_timestamp(), 'fact_order_items', {n}, '{note}')")
    print(f"snapshot '{note}': {n} rows")

snapshot("baseline")

Simulating the drop, removing 40% of the rows, then logging the new volume:

In [0]:
spark.sql(f"DELETE FROM {gt('fact_order_items')} WHERE pmod(hash(order_id), 10) < 4")
snapshot("simulated drop")

display(spark.sql(f"""
SELECT snapshot_ts, row_count, note FROM {gt('ops_volume_log')} ORDER BY snapshot_ts DESC LIMIT 5
"""))

![image_1795294700625.png](./image_1795294700625.png "image_1795294700625.png")

Restore the fact table to the version right before the DELETE

In [0]:
hist = spark.sql(f"DESCRIBE HISTORY {gt('fact_order_items')}").filter("operation = 'DELETE'").orderBy(F.desc("version")).first()
restore_to = hist["version"] - 1
spark.sql(f"RESTORE TABLE {gt('fact_order_items')} TO VERSION AS OF {restore_to}")
snapshot("restored")

In [0]:
hist = spark.sql(f"DESCRIBE HISTORY {gt('fact_order_items')}").filter("operation = 'DELETE'").orderBy(F.desc("version")).first()
restore_to = hist["version"] - 1
spark.sql(f"RESTORE TABLE {gt('fact_order_items')} TO VERSION AS OF {restore_to}")
snapshot("restored")

### Governance

In [0]:
grantee = "`account users`"

try:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {gold} TO {grantee}")
    for t in ["dim_date", "dim_customer", "dim_product", "dim_seller", "fact_order_items",
              "agg_sales_daily_state", "agg_sales_monthly_category_state"]:
        spark.sql(f"GRANT SELECT ON TABLE {gt(t)} TO {grantee}")
    print("granted")
except Exception as e:
    print("GRANT failed:", str(e)[:300])

display(spark.sql(f"SHOW GRANTS ON TABLE {gt('fact_order_items')}"))

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {gt('governance_state_access')} (user_email STRING, state STRING)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {gt('governance_pii_access')} (user_email STRING)")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {gt('rls_customer_state')}(row_state STRING)
RETURNS BOOLEAN
RETURN EXISTS (
  SELECT 1 FROM {gt('governance_state_access')} a
  WHERE a.user_email = current_user() AND (a.state = 'ALL' OR a.state = row_state)
)""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {gt('cls_mask_pii')}(val STRING)
RETURNS STRING
RETURN CASE
  WHEN EXISTS (SELECT 1 FROM {gt('governance_pii_access')} p WHERE p.user_email = current_user())
  THEN val ELSE '***MASKED***' END
""")

In [0]:
for t in ["fact_order_items", "agg_sales_daily_state", "agg_sales_monthly_category_state"]:
    try:
        spark.sql(f"ALTER TABLE {gt(t)} DROP ROW FILTER")
    except Exception:
        pass
    spark.sql(f"ALTER TABLE {gt(t)} SET ROW FILTER {gt('rls_customer_state')} ON (customer_state)")

for c in ["customer_unique_id", "zip_code_prefix"]:
    try:
        spark.sql(f"ALTER TABLE {gt('dim_customer')} ALTER COLUMN {c} DROP MASK")
    except Exception:
        pass
    spark.sql(f"ALTER TABLE {gt('dim_customer')} ALTER COLUMN {c} SET MASK {gt('cls_mask_pii')}")
print("row filter and column masks applied")

### Restricted user. Allowed to see only state SP and has no PII access

In [0]:
me = spark.sql("SELECT current_user()").first()[0]
spark.sql(f"DELETE FROM {gt('governance_state_access')} WHERE user_email = '{me}'")
spark.sql(f"DELETE FROM {gt('governance_pii_access')} WHERE user_email = '{me}'")
spark.sql(f"INSERT INTO {gt('governance_state_access')} VALUES ('{me}', 'SP')")

print("RLS: rows per state visible to", me)
display(spark.sql(f"SELECT customer_state, count(*) AS rows FROM {gt('fact_order_items')} GROUP BY customer_state"))
print("CLS: PII columns are masked")
display(spark.sql(f"SELECT customer_id, customer_unique_id, zip_code_prefix, city, state FROM {gt('dim_customer')} LIMIT 5"))

### full access

In [0]:
spark.sql(f"DELETE FROM {gt('governance_state_access')} WHERE user_email = '{me}'")
spark.sql(f"INSERT INTO {gt('governance_state_access')} VALUES ('{me}', 'ALL')")
spark.sql(f"INSERT INTO {gt('governance_pii_access')} VALUES ('{me}')")

display(spark.sql(f"SELECT customer_state, count(*) AS rows FROM {gt('fact_order_items')} GROUP BY customer_state ORDER BY rows DESC LIMIT 10"))
display(spark.sql(f"SELECT customer_id, customer_unique_id, zip_code_prefix, city, state FROM {gt('dim_customer')} LIMIT 5"))

In [0]:
display(spark.sql(f"SHOW TABLES IN {gold}"))
display(spark.sql(f"DESCRIBE DETAIL {gt('fact_order_items')}").select("name", "numFiles", "sizeInBytes"))

Link to dashboard:
https://adb-7405604651938730.10.azuredatabricks.net/dashboardsv3/01f1e5f09e5b1c00ba09c477d2e450a1/published?o=7405604651938730&f_ab3a6547%7Eperiod=_all_

Link to Genie: https://adb-7405604651938730.10.azuredatabricks.net/genie/rooms/01f1e6083e5119f6a9c93e98245c12f6?o=7405604651938730